In [58]:
import requests
import time
import pandas as pd
from typing import Optional, Dict, List
from id_converter import url_id_to_numeric, numeric_to_url_id


In [59]:
def fetch_event(event_id: int, timeout: int = 10) -> Optional[Dict]:
    """
    Fetch raw event data from the API.
    
    Returns:
        dict if valid event
        None if invalid / not found / error
    """
    str_event_id = numeric_to_url_id(event_id)
    url = f"https://results.advancedeventsystems.com/api/event/{str_event_id}"
    
    try:
        response = requests.get(url, timeout=timeout)
        
        if response.status_code != 200:
            return None
        
        data = response.json()
        
        if not data or 'EventId' not in data:
            print("No data")
            return None
        
        return data
    
    except requests.RequestException:
        return None
        

In [60]:
def parse_event(data: Dict) -> Dict:
    """
    Extract relevant fields from event data.
    """
    event_id = data.get('EventId')
    event_name = data.get('name', '')
    occurred = data.get('IsOver')
    
    divisions = data.get('divisions', [])
    division_ids = [
        d.get('divisionId') for d in divisions if 'divisionId' in d
    ]
    
    occurred = "Yes" if division_ids else "No"
    
    return {
        "event_id": event_id,
        "event_name": event_name,
        "occurred": occurred,
        "divisions": division_ids
    }

In [61]:
def process_event(event_id: int) -> Dict:
    """
    Fetch + parse + convert a single event.
    Always returns a structured record.
    """
    data = fetch_event(event_id)
    
    if data:
        parsed = parse_event(data)
    else:
        parsed = {
            "event_id": event_id,
            "event_name": "",
            "occurred": "No",
            "divisions": []
        }
    
    parsed["converted_id"] = numeric_to_url_id(event_id)
    
    return parsed

In [62]:
def process_event_range(start_id: int, end_id: int, delay: float = 0.3) -> List[Dict]:
    """
    Loop through a range of event IDs and collect results.
    """
    results = []
    
    for event_id in range(start_id, end_id + 1):
        print(f"Processing event {event_id}...")
        
        result = process_event(event_id)
        results.append(result)
        
        time.sleep(delay)
    
    return results

In [63]:
def results_to_dataframe(results: List[Dict]) -> pd.DataFrame:
    """
    Convert results list into a pandas DataFrame.
    """
    df = pd.DataFrame(results)
    
    # Convert list → string for CSV compatibility
    df['divisions'] = df['divisions'].apply(
        lambda x: ",".join(map(str, x)) if isinstance(x, list) else ""
    )
    
    return df

In [64]:
def save_results_to_csv(df: pd.DataFrame, filename: str) -> None:
    """
    Save DataFrame to CSV.
    """
    df.to_csv(filename, index=False)

In [65]:
#from aes_events import process_event_range, results_to_dataframe, save_results_to_csv

results = process_event_range(40000, 45000)

df = results_to_dataframe(results)

save_results_to_csv(df, "events.csv")

Processing event 41090...
Processing event 41091...
Processing event 41092...
Processing event 41093...
Processing event 41094...
Processing event 41095...
Processing event 41096...
Processing event 41097...
Processing event 41098...
Processing event 41099...
Processing event 41100...


In [66]:
df

,event_id,event_name,occurred,divisions,converted_id
0,41090,,No,,PTAwMDAwNDEwOTA90
1,41091,,No,,PTAwMDAwNDEwOTE90
2,41092,,No,,PTAwMDAwNDEwOTI90
3,41093,,No,,PTAwMDAwNDEwOTM90
4,41094,,No,,PTAwMDAwNDEwOTQ90
5,41095,,No,,PTAwMDAwNDEwOTU90
6,41096,,No,,PTAwMDAwNDEwOTY90
7,41097,,No,,PTAwMDAwNDEwOTc90
8,41098,,No,,PTAwMDAwNDEwOTg90
9,41099,,No,,PTAwMDAwNDEwOTk90


In [67]:
fetch_event(41091,10)

{'Key': 'PTAwMDAwNDEwOTE90',
 'EventId': 41091,
 'Name': '2026 AAU Rocky Mount Grand Prix and Boys ECPL',
 'StartDate': '2026-04-18T00:00:00',
 'EndDate': '2026-04-19T23:59:59.9999999',
 'Location': ' Rocky Mount  Events Center',
 'CustomEventType': None,
 'IsOver': True,
 'Clubs': [{'ClubId': 32710, 'Name': '495 Legacy Volleyball, Inc'},
  {'ClubId': 14541, 'Name': 'Active Volleyball Club'},
  {'ClubId': 18629, 'Name': 'Albemarle Regional Volleyball Club'},
  {'ClubId': 31017, 'Name': 'Apex'},
  {'ClubId': 24856, 'Name': 'Appalachian Volleyball Club'},
  {'ClubId': 24416, 'Name': 'Battle Volleyball Club'},
  {'ClubId': 27418, 'Name': 'Carolina Performance Volleyball Club'},
  {'ClubId': 32545, 'Name': 'Carolina Vibe Volleyball Club'},
  {'ClubId': 33046, 'Name': 'CAVB Club'},
  {'ClubId': 4571, 'Name': 'Champion Volleyball Club'},
  {'ClubId': 32583, 'Name': 'Club Boom'},
  {'ClubId': 30190, 'Name': 'Coastal Cities Volleyball Club'},
  {'ClubId': 31150, 'Name': 'Coastline Volleyball C